# 🏥 Medical LLM Benchmark Evaluation - Fixed Version
## Real Performance Testing on MedQA & PubMedQA

**This notebook performs REAL evaluation (no assumptions!):**
- ✅ Loads real models (base + fine-tuned)
- ✅ Tests on real MedQA and PubMedQA datasets
- ✅ Generates actual predictions for each question
- ✅ Calculates real metrics (Accuracy, F1, Precision, Recall)
- ✅ Supports both HuggingFace and local (Google Drive) models
- ✅ Handles LoRA adapters correctly

**Estimated Time:** 1-3 hours depending on NUM_SAMPLES

## 📦 Step 1: Install Required Libraries

In [ ]:
%%capture
# Install all required libraries
!pip install -q transformers datasets torch scikit-learn pandas numpy matplotlib seaborn
!pip install -q accelerate bitsandbytes peft
!pip install -q sentencepiece protobuf

## 📚 Step 2: Import Libraries

In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from tqdm.auto import tqdm
import json
import re
import os
from typing import List, Dict, Tuple
import warnings
import gc
warnings.filterwarnings('ignore')

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

print("✅ Libraries imported successfully!")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 🔧 Step 3: Mount Google Drive (If Using Local Models)

In [ ]:
# Mount Google Drive to access fine-tuned models
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted!")

## ⚙️ Step 4: Configuration

**IMPORTANT:** Adjust these settings based on your setup

In [ ]:
# ==================== CONFIGURATION ====================

# Test Settings
NUM_SAMPLES = 100  # Number of questions per dataset (100 = ~30 min, 500 = ~2 hours)
MAX_NEW_TOKENS = 256
TEMPERATURE = 0.1

# Output Directory
OUTPUT_DIR = './evaluation_results/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Model Configurations
# Set EVALUATE_FINETUNED = False if you only want to test baseline models
EVALUATE_FINETUNED = True  # Change to False to skip fine-tuned models

# Base Models (Always evaluated)
BASE_MODELS = {
    'llama3_base': 'unsloth/llama-3-8b-Instruct-bnb-4bit',
    'llama2_base': 'unsloth/llama-2-7b-chat-bnb-4bit',
    'mistral_base': 'unsloth/mistral-7b-instruct-v0.2-bnb-4bit',
    'gemma_base': 'unsloth/gemma-1.1-7b-it-bnb-4bit',
}

# Fine-tuned Models (Local paths from Google Drive)
# UPDATE THESE PATHS to match your Google Drive structure
FINETUNED_MODELS = {
    'llama3_medical': '/content/drive/MyDrive/graduation_project/llama3_medical_finetuned',
    'llama2_medical': '/content/drive/MyDrive/graduation_project/llama2_medical_finetuned',
    'mistral_medical': '/content/drive/MyDrive/graduation_project/mistral_medical_finetuned',
    'gemma_medical': '/content/drive/MyDrive/graduation_project/gemma_medical_finetuned',
}

# Map fine-tuned models to their base models
FINETUNED_TO_BASE = {
    'llama3_medical': 'unsloth/llama-3-8b-Instruct-bnb-4bit',
    'llama2_medical': 'unsloth/llama-2-7b-chat-bnb-4bit',
    'mistral_medical': 'unsloth/mistral-7b-instruct-v0.2-bnb-4bit',
    'gemma_medical': 'unsloth/gemma-1.1-7b-it-bnb-4bit',
}

# ========================================================

print("="*80)
print("CONFIGURATION SUMMARY")
print("="*80)
print(f"Samples per dataset: {NUM_SAMPLES}")
print(f"Evaluate fine-tuned models: {EVALUATE_FINETUNED}")
print(f"Base models: {len(BASE_MODELS)}")
if EVALUATE_FINETUNED:
    print(f"Fine-tuned models: {len(FINETUNED_MODELS)}")
print(f"Output directory: {OUTPUT_DIR}")
print("="*80)

## 🛠️ Step 5: Helper Functions

These functions handle model loading, prompt formatting, and evaluation

In [ ]:
def load_model_and_tokenizer(model_path: str, is_finetuned: bool = False, base_model_path: str = None):
    """
    Load model and tokenizer with proper handling of both HuggingFace and local models.
    
    Args:
        model_path: Path to model (HuggingFace ID or local path)
        is_finetuned: Whether this is a fine-tuned model (LoRA adapters)
        base_model_path: Base model path (required if is_finetuned=True)
    """
    print(f"\n{'='*80}")
    print(f"Loading: {model_path}")
    print(f"Fine-tuned: {is_finetuned}")
    print(f"{'='*80}")
    
    # Configure 4-bit quantization
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    
    try:
        if is_finetuned:
            # Load base model first
            print(f"Loading base model: {base_model_path}")
            model = AutoModelForCausalLM.from_pretrained(
                base_model_path,
                quantization_config=bnb_config,
                device_map="auto",
                trust_remote_code=True,
            )
            
            # Load LoRA adapters
            print(f"Loading LoRA adapters from: {model_path}")
            model = PeftModel.from_pretrained(model, model_path)
            
            # Load tokenizer from base model
            tokenizer = AutoTokenizer.from_pretrained(base_model_path, trust_remote_code=True)
            
        else:
            # Load regular HuggingFace model
            tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
            model = AutoModelForCausalLM.from_pretrained(
                model_path,
                quantization_config=bnb_config,
                device_map="auto",
                trust_remote_code=True,
            )
        
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.padding_side = "right"
        model.eval()
        
        print(f"✅ Model loaded successfully!")
        print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")
        
        return model, tokenizer
        
    except Exception as e:
        print(f"❌ Error loading model: {str(e)}")
        raise


def format_prompt(question: str, options: List[str] = None, model_type: str = "llama3") -> str:
    """
    Format prompt according to model's instruction format.
    """
    if options:
        # Multiple choice question
        options_text = "\n".join([f"{chr(65+i)}. {opt}" for i, opt in enumerate(options)])
        prompt = f"""You are a medical expert. Answer the following medical question by selecting the correct option.

Question: {question}

Options:
{options_text}

Answer with only the letter (A, B, C, or D) of the correct option."""
    else:
        # Yes/No/Maybe question
        prompt = f"""You are a medical expert. Answer the following medical question with 'yes', 'no', or 'maybe'.

Question: {question}

Answer:"""
    
    # Apply model-specific formatting
    model_type = model_type.lower()
    
    if "gemma" in model_type:
        return f"<start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n"
    elif "llama-2" in model_type or "llama2" in model_type:
        return f"[INST] {prompt} [/INST]"
    elif "llama-3" in model_type or "llama3" in model_type:
        return f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{prompt}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
    elif "mistral" in model_type:
        return f"<s>[INST] {prompt} [/INST]"
    else:
        return prompt


def generate_response(model, tokenizer, prompt: str, max_new_tokens: int = 256, temperature: float = 0.1) -> str:
    """
    Generate response from model.
    """
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True if temperature > 0 else False,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Remove the prompt from response
    if prompt in response:
        response = response.replace(prompt, "").strip()
    
    return response


def extract_answer(response: str, answer_type: str = "multiple_choice") -> str:
    """
    Extract answer from model response.
    """
    response = response.strip().upper()
    
    if answer_type == "multiple_choice":
        # Look for A, B, C, or D
        match = re.search(r'\b([A-D])\b', response)
        if match:
            return match.group(1)
        # If no match, return first character if it's A-D
        if response and response[0] in ['A', 'B', 'C', 'D']:
            return response[0]
        return "INVALID"
    else:
        # Yes/No/Maybe
        response_lower = response.lower()
        if "yes" in response_lower:
            return "yes"
        elif "no" in response_lower:
            return "no"
        elif "maybe" in response_lower:
            return "maybe"
        return "INVALID"


def calculate_metrics(predictions: List[str], labels: List[str]) -> Dict:
    """
    Calculate evaluation metrics.
    """
    # Filter out invalid predictions
    valid_indices = [i for i, p in enumerate(predictions) if p != "INVALID"]
    
    if len(valid_indices) == 0:
        return {
            'accuracy': 0.0,
            'f1_macro': 0.0,
            'f1_weighted': 0.0,
            'precision': 0.0,
            'recall': 0.0,
            'valid_responses': 0.0
        }
    
    valid_predictions = [predictions[i] for i in valid_indices]
    valid_labels = [labels[i] for i in valid_indices]
    
    return {
        'accuracy': accuracy_score(valid_labels, valid_predictions),
        'f1_macro': f1_score(valid_labels, valid_predictions, average='macro', zero_division=0),
        'f1_weighted': f1_score(valid_labels, valid_predictions, average='weighted', zero_division=0),
        'precision': precision_score(valid_labels, valid_predictions, average='weighted', zero_division=0),
        'recall': recall_score(valid_labels, valid_predictions, average='weighted', zero_division=0),
        'valid_responses': len(valid_indices) / len(predictions)
    }


print("✅ Helper functions defined successfully!")

## 📊 Step 6: Load Benchmark Datasets

In [ ]:
print("\n" + "="*80)
print("LOADING BENCHMARK DATASETS")
print("="*80)

# Load MedQA
print("\nLoading MedQA dataset...")
try:
    medqa_dataset = load_dataset("GBaker/MedQA-USMLE-4-options", split="test")
except:
    print("Trying alternative MedQA source...")
    medqa_dataset = load_dataset("bigbio/med_qa", "med_qa_en_4options_source", split="test")

if NUM_SAMPLES:
    medqa_dataset = medqa_dataset.select(range(min(NUM_SAMPLES, len(medqa_dataset))))

print(f"✅ Loaded {len(medqa_dataset)} MedQA samples")

# Load PubMedQA
print("\nLoading PubMedQA dataset...")
pubmedqa_dataset = load_dataset("pubmed_qa", "pqa_labeled", split="train")

if NUM_SAMPLES:
    pubmedqa_dataset = pubmedqa_dataset.select(range(min(NUM_SAMPLES, len(pubmedqa_dataset))))

print(f"✅ Loaded {len(pubmedqa_dataset)} PubMedQA samples")
print("\n" + "="*80)

## 🧪 Step 7: Evaluate Models on Benchmarks

**This is where REAL evaluation happens!**
- Each question is actually sent to the model
- Model generates a real response
- Answers are compared to ground truth
- Real metrics are calculated

**⏱️ This will take 1-3 hours depending on NUM_SAMPLES**

In [ ]:
def evaluate_model_on_dataset(model, tokenizer, dataset, dataset_name: str, model_type: str) -> Dict:
    """
    Evaluate a model on a benchmark dataset.
    This performs REAL evaluation - no assumptions!
    """
    print(f"\n{'='*80}")
    print(f"EVALUATING ON {dataset_name}")
    print(f"{'='*80}")
    
    predictions = []
    labels = []
    responses_log = []
    
    answer_type = "yes_no_maybe" if dataset_name == "PubMedQA" else "multiple_choice"
    
    for idx, sample in enumerate(tqdm(dataset, desc=f"Testing {dataset_name}")):
        try:
            if dataset_name == "MedQA":
                # Extract question and options
                question = sample.get('question', sample.get('Question', ''))
                options = sample.get('options', sample.get('Options', {}))
                
                if isinstance(options, dict):
                    options_list = [options.get(k, '') for k in ['A', 'B', 'C', 'D']]
                else:
                    options_list = options if isinstance(options, list) else []
                
                correct_answer = sample.get('answer', sample.get('Answer', 'A')).strip().upper()
                if len(correct_answer) > 1:
                    correct_answer = correct_answer[0]
                
                prompt = format_prompt(question, options_list, model_type)
                
            else:  # PubMedQA
                question = sample['question']
                correct_answer = sample['final_decision'].lower()
                prompt = format_prompt(question, None, model_type)
            
            # REAL INFERENCE - This actually queries the model!
            response = generate_response(model, tokenizer, prompt, MAX_NEW_TOKENS, TEMPERATURE)
            predicted_answer = extract_answer(response, answer_type)
            
            predictions.append(predicted_answer)
            labels.append(correct_answer)
            
            responses_log.append({
                'question': question,
                'predicted': predicted_answer,
                'correct': correct_answer,
                'response': response[:200]
            })
            
            # Progress update every 25 questions
            if (idx + 1) % 25 == 0:
                temp_metrics = calculate_metrics(predictions, labels)
                print(f"\n  Progress [{idx + 1}/{len(dataset)}]:")
                print(f"    Accuracy: {temp_metrics['accuracy']:.4f} ({temp_metrics['accuracy']*100:.2f}%)")
                print(f"    Valid Responses: {temp_metrics['valid_responses']:.2%}")
        
        except Exception as e:
            print(f"\n  ⚠️ Error on question {idx}: {str(e)}")
            predictions.append("INVALID")
            labels.append(sample.get('answer', sample.get('final_decision', 'A')))
            continue
    
    # Calculate final metrics
    metrics = calculate_metrics(predictions, labels)
    
    print(f"\n{'='*80}")
    print(f"FINAL RESULTS - {dataset_name}:")
    print(f"{'='*80}")
    print(f"Accuracy:        {metrics['accuracy']:.4f} ({metrics['accuracy']*100:.2f}%)")
    print(f"F1 Score (Macro):     {metrics['f1_macro']:.4f}")
    print(f"F1 Score (Weighted):  {metrics['f1_weighted']:.4f}")
    print(f"Precision:       {metrics['precision']:.4f}")
    print(f"Recall:          {metrics['recall']:.4f}")
    print(f"Valid Responses: {metrics['valid_responses']:.2%}")
    print(f"{'='*80}")
    
    return {
        'metrics': metrics,
        'predictions': predictions,
        'labels': labels,
        'responses_log': responses_log
    }


# Storage for all results
all_results = {}

print("✅ Evaluation function ready!")

## 🔵 Step 8: Evaluate BASE Models

**Testing baseline models (no fine-tuning)**

In [ ]:
print("\n" + "#"*80)
print("# PHASE 1: EVALUATING BASE MODELS")
print("#"*80)

base_results = {}

for model_key, model_path in BASE_MODELS.items():
    print(f"\n\n{'='*80}")
    print(f"MODEL: {model_key}")
    print(f"PATH: {model_path}")
    print(f"{'='*80}")
    
    # Load model
    model, tokenizer = load_model_and_tokenizer(model_path, is_finetuned=False)
    
    # Determine model type for prompt formatting
    model_type = model_key.split('_')[0]
    
    # Evaluate on MedQA
    medqa_results = evaluate_model_on_dataset(
        model, tokenizer, medqa_dataset, "MedQA", model_type
    )
    
    # Evaluate on PubMedQA
    pubmedqa_results = evaluate_model_on_dataset(
        model, tokenizer, pubmedqa_dataset, "PubMedQA", model_type
    )
    
    base_results[model_key] = {
        'MedQA': medqa_results,
        'PubMedQA': pubmedqa_results
    }
    
    print(f"\n✅ Completed evaluation for {model_key}")
    
    # Clean up GPU memory
    del model
    del tokenizer
    torch.cuda.empty_cache()
    gc.collect()

print("\n" + "#"*80)
print("# BASE MODEL EVALUATION COMPLETE!")
print("#"*80)

## 🟢 Step 9: Evaluate FINE-TUNED Models

**Testing fine-tuned models (with LoRA adapters)**

In [ ]:
if EVALUATE_FINETUNED:
    print("\n" + "#"*80)
    print("# PHASE 2: EVALUATING FINE-TUNED MODELS")
    print("#"*80)
    
    finetuned_results = {}
    
    for model_key, model_path in FINETUNED_MODELS.items():
        print(f"\n\n{'='*80}")
        print(f"MODEL: {model_key}")
        print(f"PATH: {model_path}")
        print(f"{'='*80}")
        
        # Check if path exists
        if not os.path.exists(model_path):
            print(f"⚠️ WARNING: Path does not exist: {model_path}")
            print(f"Skipping {model_key}...")
            continue
        
        try:
            # Get base model path
            base_model_path = FINETUNED_TO_BASE[model_key]
            
            # Load model with LoRA adapters
            model, tokenizer = load_model_and_tokenizer(
                model_path, 
                is_finetuned=True, 
                base_model_path=base_model_path
            )
            
            # Determine model type
            model_type = model_key.split('_')[0]
            
            # Evaluate on MedQA
            medqa_results = evaluate_model_on_dataset(
                model, tokenizer, medqa_dataset, "MedQA", model_type
            )
            
            # Evaluate on PubMedQA
            pubmedqa_results = evaluate_model_on_dataset(
                model, tokenizer, pubmedqa_dataset, "PubMedQA", model_type
            )
            
            finetuned_results[model_key] = {
                'MedQA': medqa_results,
                'PubMedQA': pubmedqa_results
            }
            
            print(f"\n✅ Completed evaluation for {model_key}")
            
        except Exception as e:
            print(f"\n❌ Error evaluating {model_key}: {str(e)}")
            print(f"Skipping this model...")
            continue
        
        finally:
            # Clean up GPU memory
            if 'model' in locals():
                del model
            if 'tokenizer' in locals():
                del tokenizer
            torch.cuda.empty_cache()
            gc.collect()
    
    print("\n" + "#"*80)
    print("# FINE-TUNED MODEL EVALUATION COMPLETE!")
    print("#"*80)
    
    # Merge results
    all_results = {**base_results, **finetuned_results}
else:
    print("\n⏩ Skipping fine-tuned model evaluation (EVALUATE_FINETUNED = False)")
    all_results = base_results

## 📊 Step 10: Generate Comparison Report

In [ ]:
print("\n" + "="*120)
print("BASELINE MODEL PERFORMANCE SUMMARY")
print("="*120)

baseline_data = []

for model_key in base_results.keys():
    model_name = model_key.replace('_base', '').upper()
    
    medqa_acc = base_results[model_key]['MedQA']['metrics']['accuracy']
    medqa_f1 = base_results[model_key]['MedQA']['metrics']['f1_weighted']
    
    pubmed_acc = base_results[model_key]['PubMedQA']['metrics']['accuracy']
    pubmed_f1 = base_results[model_key]['PubMedQA']['metrics']['f1_weighted']
    
    baseline_data.append({
        'Model': model_name,
        'MedQA Accuracy': f"{medqa_acc*100:.2f}%",
        'MedQA F1': f"{medqa_f1:.4f}",
        'PubMedQA Accuracy': f"{pubmed_acc*100:.2f}%",
        'PubMedQA F1': f"{pubmed_f1:.4f}"
    })

baseline_df = pd.DataFrame(baseline_data)
print(baseline_df.to_string(index=False))
print("="*120)

# Save baseline results
baseline_df.to_csv(f"{OUTPUT_DIR}/baseline_results.csv", index=False)
print(f"\n✅ Saved baseline results to {OUTPUT_DIR}/baseline_results.csv")

# If fine-tuned models were evaluated, create comparison
if EVALUATE_FINETUNED and len(finetuned_results) > 0:
    print("\n" + "="*120)
    print("BASE vs FINE-TUNED COMPARISON")
    print("="*120)
    
    comparison_data = []
    
    for model_name in ['llama3', 'llama2', 'mistral', 'gemma']:
        base_key = f"{model_name}_base"
        ft_key = f"{model_name}_medical"
        
        if base_key in base_results and ft_key in finetuned_results:
            for dataset_name in ['MedQA', 'PubMedQA']:
                base_acc = base_results[base_key][dataset_name]['metrics']['accuracy']
                ft_acc = finetuned_results[ft_key][dataset_name]['metrics']['accuracy']
                improvement = ((ft_acc - base_acc) / base_acc * 100) if base_acc > 0 else 0
                
                comparison_data.append({
                    'Model': model_name.upper(),
                    'Dataset': dataset_name,
                    'Base Accuracy': f"{base_acc*100:.2f}%",
                    'Fine-tuned Accuracy': f"{ft_acc*100:.2f}%",
                    'Improvement': f"{improvement:+.2f}%"
                })
    
    if comparison_data:
        comparison_df = pd.DataFrame(comparison_data)
        print(comparison_df.to_string(index=False))
        print("="*120)
        
        comparison_df.to_csv(f"{OUTPUT_DIR}/comparison_results.csv", index=False)
        print(f"\n✅ Saved comparison to {OUTPUT_DIR}/comparison_results.csv")

## 📈 Step 11: Create Visualizations

In [ ]:
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Extract model names
model_names = [key.replace('_base', '').upper() for key in base_results.keys()]

# Baseline performance
medqa_base = [base_results[key]['MedQA']['metrics']['accuracy'] * 100 for key in base_results.keys()]
pubmed_base = [base_results[key]['PubMedQA']['metrics']['accuracy'] * 100 for key in base_results.keys()]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Baseline Model Performance Comparison (REAL EVALUATION RESULTS)', fontsize=14, fontweight='bold')

x = np.arange(len(model_names))

# MedQA
axes[0].bar(x, medqa_base, color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A'], alpha=0.8)
axes[0].set_xlabel('Model', fontweight='bold')
axes[0].set_ylabel('Accuracy (%)', fontweight='bold')
axes[0].set_title('MedQA Performance', fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(model_names, rotation=45)
axes[0].grid(True, alpha=0.3, axis='y')

for i, v in enumerate(medqa_base):
    axes[0].text(i, v + 1, f'{v:.1f}%', ha='center', va='bottom', fontweight='bold')

# PubMedQA
axes[1].bar(x, pubmed_base, color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A'], alpha=0.8)
axes[1].set_xlabel('Model', fontweight='bold')
axes[1].set_ylabel('Accuracy (%)', fontweight='bold')
axes[1].set_title('PubMedQA Performance', fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(model_names, rotation=45)
axes[1].grid(True, alpha=0.3, axis='y')

for i, v in enumerate(pubmed_base):
    axes[1].text(i, v + 1, f'{v:.1f}%', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/baseline_comparison.png", dpi=300, bbox_inches='tight')
print(f"\n✅ Saved visualization to {OUTPUT_DIR}/baseline_comparison.png")
plt.show()

# If fine-tuned models evaluated, create comparison plot
if EVALUATE_FINETUNED and len(finetuned_results) > 0:
    # Create comparison visualization here
    print("\n📊 Fine-tuned comparison visualization can be added if needed")

## 💾 Step 12: Save Detailed Results

In [ ]:
# Save all results to JSON
results_summary = {
    'configuration': {
        'num_samples': NUM_SAMPLES,
        'max_new_tokens': MAX_NEW_TOKENS,
        'temperature': TEMPERATURE,
        'evaluated_finetuned': EVALUATE_FINETUNED
    },
    'base_results': {},
    'finetuned_results': {}
}

# Add base model results (metrics only, not full responses)
for key, results in base_results.items():
    results_summary['base_results'][key] = {
        'MedQA': results['MedQA']['metrics'],
        'PubMedQA': results['PubMedQA']['metrics']
    }

# Add fine-tuned results if available
if EVALUATE_FINETUNED and 'finetuned_results' in locals():
    for key, results in finetuned_results.items():
        results_summary['finetuned_results'][key] = {
            'MedQA': results['MedQA']['metrics'],
            'PubMedQA': results['PubMedQA']['metrics']
        }

with open(f"{OUTPUT_DIR}/evaluation_results.json", 'w') as f:
    json.dump(results_summary, f, indent=2)

print(f"\n✅ Saved detailed results to {OUTPUT_DIR}/evaluation_results.json")

# Save sample predictions for inspection
for key in base_results.keys():
    sample_predictions = base_results[key]['MedQA']['responses_log'][:10]
    with open(f"{OUTPUT_DIR}/{key}_sample_predictions.json", 'w') as f:
        json.dump(sample_predictions, f, indent=2)

print(f"\n✅ Saved sample predictions for inspection")

## 📥 Step 13: Download Results

In [ ]:
from google.colab import files

print("📥 Downloading results files...\n")

# Download main results
files.download(f'{OUTPUT_DIR}/baseline_results.csv')
files.download(f'{OUTPUT_DIR}/baseline_comparison.png')
files.download(f'{OUTPUT_DIR}/evaluation_results.json')

if EVALUATE_FINETUNED and os.path.exists(f'{OUTPUT_DIR}/comparison_results.csv'):
    files.download(f'{OUTPUT_DIR}/comparison_results.csv')

print("\n✅ Download complete!")

## 🎉 EVALUATION COMPLETE!

### What We Did:
✅ Loaded real medical benchmark datasets (MedQA & PubMedQA)  
✅ Tested models on real questions (no assumptions!)  
✅ Generated actual predictions for each question  
✅ Calculated real metrics (Accuracy, F1, Precision, Recall)  
✅ Created comparison reports and visualizations  
✅ Saved all results for your graduation project  

### Results Location:
- CSV files: `./evaluation_results/*.csv`
- Visualizations: `./evaluation_results/*.png`
- JSON results: `./evaluation_results/*.json`

### Next Steps:
1. Review the baseline performance
2. If fine-tuned models were evaluated, compare improvements
3. Use these results in your graduation project report
4. Cite the real metrics in your thesis!

**🎓 This evaluation used REAL data, REAL models, and REAL calculations - no assumptions!**